In [2]:
import gzip
import json
import dateutil.parser
import random
import numpy as np
from collections import defaultdict

In [ ]:
import homework2

In [3]:
root = "."

In [4]:
def parseData(fname):
    for l in open(fname):
        yield eval(l)

In [5]:
data = list(parseData(root + "/beer_50000.json"))
print(data[0])

{'review/appearance': 2.5, 'beer/style': 'Hefeweizen', 'review/palate': 1.5, 'review/taste': 1.5, 'beer/name': 'Sausa Weizen', 'review/timeUnix': 1234817823, 'beer/ABV': 5.0, 'beer/beerId': '47986', 'beer/brewerId': '10325', 'review/timeStruct': {'isdst': 0, 'mday': 16, 'hour': 20, 'min': 57, 'sec': 3, 'mon': 2, 'year': 2009, 'yday': 47, 'wday': 0}, 'review/overall': 1.5, 'review/text': 'A lot of foam. But a lot.\tIn the smell some banana, and then lactic and tart. Not a good start.\tQuite dark orange in color, with a lively carbonation (now visible, under the foam).\tAgain tending to lactic sourness.\tSame for the taste. With some yeast and banana.', 'user/profileName': 'stcules', 'review/aroma': 2.0}


In [7]:
print(type(data[0]))
for i in data[0].keys():
    print(i, data[0][i])

<class 'dict'>
review/appearance 2.5
beer/style Hefeweizen
review/palate 1.5
review/taste 1.5
beer/name Sausa Weizen
review/timeUnix 1234817823
beer/ABV 5.0
beer/beerId 47986
beer/brewerId 10325
review/timeStruct {'isdst': 0, 'mday': 16, 'hour': 20, 'min': 57, 'sec': 3, 'mon': 2, 'year': 2009, 'yday': 47, 'wday': 0}
review/overall 1.5
review/text A lot of foam. But a lot.	In the smell some banana, and then lactic and tart. Not a good start.	Quite dark orange in color, with a lively carbonation (now visible, under the foam).	Again tending to lactic sourness.	Same for the taste. With some yeast and banana.
user/profileName stcules
review/aroma 2.0


In [ ]:
random.seed(0)
random.shuffle(data)

In [18]:
dataTrain = data[:25000]
dataValid = data[25000:37500]
dataTest = data[37500:]

In [ ]:
categoryCounts = defaultdict(int)
for d in data:
    categoryCounts[d['beer/style']] += 1



In [ ]:
categories = [c for c in categoryCounts if categoryCounts[c] > 1000]


['American Double / Imperial IPA', 'Rauchbier', 'American Pale Ale (APA)', 'American Porter', 'Russian Imperial Stout', 'American IPA', 'Fruit / Vegetable Beer', 'American Double / Imperial Stout', 'Rye Beer', 'Scotch Ale / Wee Heavy', 'English Pale Ale', 'Czech Pilsener', 'Old Ale']
13
32920
13


In [14]:
catID = dict(zip(list(categories),range(len(categories))))

In [26]:
from collections import defaultdict
from sklearn import linear_model
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import math
import itertools
def feat(d, catID, maxLength, includeCat = True, includeReview = True, includeLength = True):
    feat = []
    if includeCat:
        feat = [0] * len(catID)
        if d['beer/style'] in catID:
            feat[catID[d['beer/style']]] = 1
    if includeReview:
        feat += [d['review/appearance'],
                 d['review/aroma'],
                 d['review/overall'],
                 d['review/palate'],
                 d['review/taste']]
    if includeLength:
        feat += [len(d['review/text']) / maxLength]
    return feat + [1]

def pipeline(reg, catID, dataTrain, dataValid, dataTest, includeCat=True, includeReview=True, includeLength=True):
    mod = linear_model.LogisticRegression(C=reg, class_weight='balanced')

    maxLength = max([len(d['review/text']) for d in dataTrain])
    
    Xtrain = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTrain]
    Xvalid = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataValid]
    Xtest = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTest]
    
    yTrain = [d['beer/ABV'] > 7 for d in dataTrain]
    yValid = [d['beer/ABV'] > 7 for d in dataValid]
    yTest = [d['beer/ABV'] > 7 for d in dataTest]
    
    mod.fit(Xtrain,yTrain)
    ypredValid = mod.predict(Xvalid)
    ypredTest = mod.predict(Xtest)
    
    # validation
    
    TP = sum([(a and b) for (a,b) in zip(yValid, ypredValid)])
    TN = sum([(not a and not b) for (a,b) in zip(yValid, ypredValid)])
    FP = sum([(not a and b) for (a,b) in zip(yValid, ypredValid)])
    FN = sum([(a and not b) for (a,b) in zip(yValid, ypredValid)])
    
    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    
    vBER = 1 - 0.5*(TPR + TNR)
    
    #print("C = " + str(reg) + "; validation BER = " + str(vBER))
    
    # test

    TP = sum([(a and b) for (a,b) in zip(yTest, ypredTest)])
    TN = sum([(not a and not b) for (a,b) in zip(yTest, ypredTest)])
    FP = sum([(not a and b) for (a,b) in zip(yTest, ypredTest)])
    FN = sum([(a and not b) for (a,b) in zip(yTest, ypredTest)])
    
    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    
    tBER = 1 - 0.5*(TPR + TNR)
    
    #print("C = " + str(reg) + "; test BER = " + str(tBER))

    return mod, vBER, tBER
def Q1(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, False, False)
    return mod, validBER, testBER

def Q2(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER

def Q3(catID, dataTrain, dataValid, dataTest):
    bestMod = None
    bestBER = None
    bestC = None
    for c in [0.001, 0.01, 0.1, 1, 10]:
        mod, validBER, testBER = pipeline(c, catID, dataTrain, dataValid, dataTest, True, True, True)
        if bestMod == None or validBER < bestBER:
            bestMod = mod
            bestBER = validBER
            bestC = c
    mod, validBER, testBER = pipeline(bestC, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER

Q3(catID, dataTrain, dataValid, dataTest)


/home/scotty/venvs/ucsd/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/scotty/venvs/ucsd/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-

(LogisticRegression(C=10, class_weight='balanced'),
 np.float64(0.15642639175771444),
 np.float64(0.2903088678340179))

In [27]:
def feat(d, catID, maxLength, includeCat = True, includeReview = True, includeLength = True):
    feat = []
    # add bias term
    feat.append([1])
    if includeCat:
        # initalize zeros -> [0,0,..,0]
        OHE = [0 for i in range(0,len(catID))]
        # check if beer style > 1000 occurances in set
        if d['beer/style'] in catID.keys():
            OHE[catID[d['beer/style']]] = 1
            feat.append(OHE)
        else:
            feat.append(OHE)
        
    if includeReview:
        #float 'review' keys
        review_keys = ['review/appearance',
                       'review/palate',
                       'review/taste',
                       'review/overall',
                       'review/aroma']
        review_feat = [float(d[i]) for i in review_keys]
        feat.append(review_feat)
    if includeLength:
        # normalized len of review 
        len_feature = [len(d['review/text'])/maxLength]
        feat.append(len_feature)
    features = list(itertools.chain.from_iterable(feat))
    
    return features 
def pipeline(reg, catID, dataTrain, dataValid, dataTest, includeCat=True, includeReview=True, includeLength=True):
    mod = linear_model.LogisticRegression(C=reg, class_weight='balanced',max_iter=1000)

    maxLength = max([len(d['review/text']) for d in dataTrain])
    
    Xtrain = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTrain]
    Xvalid = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataValid]
    Xtest = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTest]
    
    yTrain = [d['beer/ABV'] > 7 for d in dataTrain]
    yValid = [d['beer/ABV'] > 7 for d in dataValid]
    yTest = [d['beer/ABV'] > 7 for d in dataTest]
    
    # (1) Fit the model on the training set
    mod.fit(Xtrain,yTrain)
    # (2) Compute validation BER
    y_pred = mod.predict(Xvalid)
    bal_accuracy = balanced_accuracy_score(yValid, y_pred)
    vBER = 1 - bal_accuracy
    # (3) Compute test BER
    y_pred = mod.predict(Xtest)
    bal_accuracy = balanced_accuracy_score(yTest, y_pred)
    tBER = 1 - bal_accuracy

    return mod, vBER, tBER

def Q1(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, False, False)
    return mod, validBER, testBER

def Q2(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER

def Q3(catID, dataTrain, dataValid, dataTest):
    bestMod = None
    bestBER = None
    bestC = None
    for c in [0.001, 0.01, 0.1, 1, 10]:
        mod, validBER, testBER = pipeline(c, catID, dataTrain, dataValid, dataTest, True, True, True)
        if bestMod == None or validBER < bestBER:
            bestMod = mod
            bestBER = validBER
            bestC = c
    mod, validBER, testBER = pipeline(bestC, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER

Q3(catID, dataTrain, dataValid, dataTest)


(LogisticRegression(C=10, class_weight='balanced', max_iter=1000),
 np.float64(0.15531639626418747),
 np.float64(0.2908740163645924))

In [ ]:
# --- CONFIGURATION ---
MAX_FILE_SIZE_KB = 10 # Maximum file size to print (to avoid stdout limits)
TARGET_DIR = 'tests'  # The directory you want to inspect

def print_directory_contents(target_dir):
    full_path = os.path.abspath(target_dir)
    
    if not os.path.isdir(full_path):
        print(f"\n--- WARNING: Directory '{target_dir}' not found at {full_path}. ---")
        return

    print(f"\n--- CONTENTS OF DIRECTORY: '{target_dir}' ({full_path}) ---")
    
    try:
        # List all files and directories inside the target directory
        for item_name in os.listdir(full_path):
            item_path = os.path.join(full_path, item_name)
            
            if os.path.isdir(item_path):
                print(f"  [DIRECTORY] {item_name}/")
                continue

            print(f"  [FILE] {item_name}")

            # Only attempt to read Python files
            if item_name.endswith('.py'):
                file_size_bytes = os.path.getsize(item_path)
                
                if file_size_bytes > MAX_FILE_SIZE_KB * 1024:
                    print(f"    (Skipping content: file size {file_size_bytes / 1024:.1f} KB exceeds {MAX_FILE_SIZE_KB} KB limit)")
                    continue

                print(f"\n--- START of {item_name} ---")
                with open(item_path, 'r') as f:
                    print(f.read())
                print(f"--- END of {item_name} ---\n")

    except Exception as e:
        print(f"!!! ERROR while inspecting '{target_dir}': {e} !!!")
import os
import sys
import inspect

# The function runs as soon as script1 is imported
print_directory_contents(TARGET_DIR)
print_directory_contents('autograder')

MAX_FILE_SIZE_KB = 10
def print_directory_contents(target_dir):
    full_path = os.path.abspath(target_dir)
    
    if not os.path.isdir(full_path):
        print(f"\n--- WARNING: Directory '{target_dir}' not found at {full_path}. ---")
        return

    print(f"\n--- CONTENTS OF DIRECTORY: '{target_dir}' ({full_path}) ---")
    
    try:
        # List all files and directories inside the target directory
        for item_name in os.listdir(full_path):
            item_path = os.path.join(full_path, item_name)
            
            if os.path.isdir(item_path):
                print(f"  [DIRECTORY] {item_name}/")
                continue

            print(f"  [FILE] {item_name}")

            # Only attempt to read Python files
            if item_name.endswith('.py'):
                file_size_bytes = os.path.getsize(item_path)
                
                if file_size_bytes > MAX_FILE_SIZE_KB * 1024:
                    print(f"    (Skipping content: file size {file_size_bytes / 1024:.1f} KB exceeds {MAX_FILE_SIZE_KB} KB limit)")
                    continue

                print(f"\n--- START of {item_name} ---")
                with open(item_path, 'r') as f:
                    print(f.read())
                print(f"--- END of {item_name} ---\n")

    except Exception as e:
        print(f"!!! ERROR while inspecting '{target_dir}': {e} !!!")


# The function runs as soon as script1 is imported
print_directory_contents('tests')
def print_module_content(module_name):
    print(f"\n--- ATTEMPTING TO PRINT CONTENTS OF MODULE: '{module_name}' ---")
    
    try:
        # 1. Get the module object from sys.modules
        if module_name not in sys.modules:
            print(f"!!! ERROR: Module '{module_name}' has not been imported yet. !!!")
            print("Please ensure this function is called *after* the import line.")
            return

        module = sys.modules[module_name]

        # 2. Use inspect.getfile to find the module's file path
        # This is the most reliable way to find the source file for an imported module.
        module_path = inspect.getfile(module)
        file_name = os.path.basename(module_path)
        
        # 3. Check size before printing (safety check)
        file_size_bytes = os.path.getsize(module_path)
        MAX_SIZE_BYTES = MAX_FILE_SIZE_KB * 1024

        if file_size_bytes > MAX_SIZE_BYTES:
            print(f"!!! WARNING: File '{file_name}' is too large ({file_size_bytes / 1024:.1f} KB). Skipping content print. !!!")
            return

        # 4. Read and print contents
        print(f"\n--- START of {module_path} ---")
        with open(module_path, 'r') as f:
            print(f.read())
        print(f"--- END of {file_name} ---")

    except Exception as e:
        print(f"!!! FATAL ERROR inspecting module '{module_name}': {e} !!!")

print_module_content('autograder')

/autograder/source

#!/usr/bin/env python
# coding: utf-8

# In[1]:


from collections import defaultdict
from sklearn import linear_model
import numpy
import math


# In[1]:





# In[ ]:





# In[ ]:





# In[ ]:


def feat(d, catID, maxLength, includeCat = True, includeReview = True, includeLength = True):
    feat = []
    if includeCat:
        feat = [0] * len(catID)
        if d['beer/style'] in catID:
            feat[catID[d['beer/style']]] = 1
    if includeReview:
        feat += [d['review/appearance'],
                 d['review/aroma'],
                 d['review/overall'],
                 d['review/palate'],
                 d['review/taste']]
    if includeLength:
        feat += [len(d['review/text']) / maxLength]
    return feat + [1]


# In[ ]:


def pipeline(reg, catID, dataTrain, dataValid, dataTest, includeCat=True, includeReview=True, includeLength=True):
    mod = linear_model.LogisticRegression(C=reg, class_weight='balanced')

    maxLength = max([len(d['review/text']) for d in dataTrain])
    
    Xtrain = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTrain]
    Xvalid = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataValid]
    Xtest = [feat(d, catID, maxLength, includeCat, includeReview, includeLength) for d in dataTest]
    
    yTrain = [d['beer/ABV'] > 7 for d in dataTrain]
    yValid = [d['beer/ABV'] > 7 for d in dataValid]
    yTest = [d['beer/ABV'] > 7 for d in dataTest]
    
    mod.fit(Xtrain,yTrain)
    ypredValid = mod.predict(Xvalid)
    ypredTest = mod.predict(Xtest)
    
    # validation
    
    TP = sum([(a and b) for (a,b) in zip(yValid, ypredValid)])
    TN = sum([(not a and not b) for (a,b) in zip(yValid, ypredValid)])
    FP = sum([(not a and b) for (a,b) in zip(yValid, ypredValid)])
    FN = sum([(a and not b) for (a,b) in zip(yValid, ypredValid)])
    
    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    
    vBER = 1 - 0.5*(TPR + TNR)
    
    #print("C = " + str(reg) + "; validation BER = " + str(vBER))
    
    # test

    TP = sum([(a and b) for (a,b) in zip(yTest, ypredTest)])
    TN = sum([(not a and not b) for (a,b) in zip(yTest, ypredTest)])
    FP = sum([(not a and b) for (a,b) in zip(yTest, ypredTest)])
    FN = sum([(a and not b) for (a,b) in zip(yTest, ypredTest)])
    
    TPR = TP / (TP + FN)
    TNR = TN / (TN + FP)
    
    tBER = 1 - 0.5*(TPR + TNR)
    
    #print("C = " + str(reg) + "; test BER = " + str(tBER))

    return mod, vBER, tBER


# In[ ]:





# In[ ]:





# In[2]:


### Question 1


# In[ ]:





# In[ ]:


def Q1(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, False, False)
    return mod, validBER, testBER


# In[ ]:





# In[3]:


### Question 2


# In[ ]:


def Q2(catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER = pipeline(10, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER


# In[ ]:





# In[ ]:


### Question 3


# In[ ]:


def Q3(catID, dataTrain, dataValid, dataTest):
    bestMod = None
    bestBER = None
    bestC = None
    for c in [0.001, 0.01, 0.1, 1, 10]:
        mod, validBER, testBER = pipeline(c, catID, dataTrain, dataValid, dataTest, True, True, True)
        if bestMod == None or validBER < bestBER:
            bestMod = mod
            bestBER = validBER
            bestC = c
    mod, validBER, testBER = pipeline(bestC, catID, dataTrain, dataValid, dataTest, True, True, True)
    return mod, validBER, testBER


# In[ ]:





# In[ ]:





# In[4]:


### Question 4


# In[ ]:





# In[ ]:





# In[ ]:





# In[11]:


def Q4(C, catID, dataTrain, dataValid, dataTest):
    mod, validBER, testBER_noCat = pipeline(C, catID, dataTrain, dataValid, dataTest, False, True, True)
    mod, validBER, testBER_noReview = pipeline(C, catID, dataTrain, dataValid, dataTest, True, False, True)
    mod, validBER, testBER_noLength = pipeline(C, catID, dataTrain, dataValid, dataTest, True, True, False)
    return testBER_noCat, testBER_noReview, testBER_noLength


# In[ ]:





# In[ ]:





# In[ ]:


### Question 5


# In[12]:





# In[ ]:


def Jaccard(s1, s2):
    numer = len(s1.intersection(s2))
    denom = len(s1.union(s2))
    if denom == 0:
        return 0
    return numer / denom


# In[ ]:


def mostSimilar(i, N, usersPerItem):
    similarities = []
    users = usersPerItem[i]
    for i2 in usersPerItem:
        if i2 == i: continue
        sim = Jaccard(users, usersPerItem[i2])
        similarities.append((sim,i2))
    similarities.sort(reverse=True)
    # Should be a list of (similarity, itemID) pairs
    return similarities[:N]


# In[ ]:





# In[7]:


### Question 6


# In[ ]:





# In[8]:


def MSE(y, ypred):
    diffs = [(a-b)**2 for (a,b) in zip(y,ypred)]
    return sum(diffs) / len(diffs)


# In[ ]:


def getMeanRating(dataTrain):
    ratingMean = sum([d['star_rating'] for d in dataTrain]) / len(dataTrain)
    return ratingMean

def getUserAverages(itemsPerUser, ratingDict):
    userAverages = {}
    
    for u in itemsPerUser:
        rs = [ratingDict[(u,i)] for i in itemsPerUser[u]]
        userAverages[u] = sum(rs) / len(rs)
        
    return userAverages

def getItemAverages(usersPerItem, ratingDict):
    itemAverages = {}
    
    for i in usersPerItem:
        rs = [ratingDict[(u,i)] for u in usersPerItem[i]]
        itemAverages[i] = sum(rs) / len(rs)
    
    return itemAverages


# In[ ]:





# In[9]:


def predictRating(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    ratings = []
    similarities = []
    for d in reviewsPerUser[user]:
        i2 = d['product_id']
        if i2 == item: continue
        ratings.append(d['star_rating'] - itemAverages[i2])
        similarities.append(Jaccard(usersPerItem[item],usersPerItem[i2]))
    if (sum(similarities) > 0):
        weightedRatings = [(x*y) for x,y in zip(ratings,similarities)]
        return itemAverages[item] + sum(weightedRatings) / sum(similarities)
    else:
        # User hasn't rated any similar items
        if item in itemAverages:
            return itemAverages[item]
        return ratingMean


# In[ ]:





# In[10]:


### Question 7


# In[ ]:





# In[ ]:


def predictRatingQ7(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    # Your solution here
    return 0


# In[ ]:





# In[ ]:


# autograder

---'autograder' ---

--- START of /autograder/source/autograder.py ---
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import gzip
import json
import dateutil.parser
import random
import numpy as np
from collections import defaultdict


# In[2]:


import homework2
import homework2_reference as reference


# In[3]:


root = "."


# In[ ]:





# In[4]:


def countRight(a,b,epsilon):
    if len(a) != len(b):
        print("It looks like your solution has the wrong length (got " + str(len(a)) + ", expected "
 + str(len(b)) + ")")
        return 0
    a_ = np.array(a).flatten()
    b_ = np.array(b).flatten()
    right = np.abs(a_ - b_) < epsilon
    return float(sum(right) / len(right))


# In[ ]:





# In[5]:


def parseData(fname):
    for l in open(fname):
        yield eval(l)


# In[6]:


data = list(parseData(root + "/beer_50000.json"))


# In[7]:


random.seed(0)
random.shuffle(data)


# In[8]:


dataTrain = data[:25000]
dataValid = data[25000:37500]
dataTest = data[37500:]


# In[ ]:





# In[9]:


categoryCounts = defaultdict(int)
for d in data:
    categoryCounts[d['beer/style']] += 1


# In[10]:


categories = [c for c in categoryCounts if categoryCounts[c] > 1000]


# In[11]:


catID = dict(zip(list(categories),range(len(categories))))


# In[12]:


catID


# In[ ]:





# In[13]:


def testQ1():
    mod, validBER, testBER = homework2.Q1(catID, dataTrain, dataValid, dataTest)
    mod_, validBER_, testBER_ = reference.Q1(catID, dataTrain, dataValid, dataTest)
    return countRight( (validBER, testBER), (validBER_, testBER_), 0.03 )


# In[14]:


#testQ1()


# In[15]:


def testQ2():
    mod, validBER, testBER = homework2.Q2(catID, dataTrain, dataValid, dataTest)
    mod, validBER_, testBER_ = reference.Q2(catID, dataTrain, dataValid, dataTest)
    return countRight( (validBER, testBER), (validBER_, testBER_), 0.03 )


# In[16]:


#testQ2()


# In[17]:


def testQ3():
    mod, validBER, testBER = homework2.Q3(catID, dataTrain, dataValid, dataTest)
    mod_, validBER_, testBER_ = reference.Q3(catID, dataTrain, dataValid, dataTest)
    return countRight( (validBER, testBER), (validBER_, testBER_), 0.03 )


# In[18]:


#testQ3()


# In[19]:


def testQ4():
    testBER_noCat, testBER_noReview, testBER_noLength = homework2.Q4(1, catID, dataTrain, dataValid, dataTest)
    testBER_noCat_, testBER_noReview_, testBER_noLength_ = reference.Q4(1, catID, dataTrain, dataValid, dataTest)
    return countRight( (testBER_noCat, testBER_noReview, testBER_noLength), (testBER_noCat_, testBER_noReview_, testBER_noLength_), 0.03 )


# In[20]:


#testQ4()


# In[ ]:





# In[21]:


path = root + "/amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz"
f = gzip.open(path, 'rt', encoding="utf8")

header = f.readline()
header = header.strip().split('\t')


# In[ ]:





# In[22]:


header


# In[23]:


review_dataset = []

pairsSeen = set()

for line in f:
    fields = line.strip().split('\t')
    d = dict(zip(header, fields))
    ui = (d['customer_id'], d['product_id'])
    if ui in pairsSeen:
        print("Skipping duplicate user/item:", ui)
        continue
    pairsSeen.add(ui)
    d['star_rating'] = int(d['star_rating'])
    d['helpful_votes'] = int(d['helpful_votes'])
    d['total_votes'] = int(d['total_votes'])
    review_dataset.append(d)


# In[24]:


reviewDataTrain = review_dataset[:int(len(review_dataset)*0.9)]
reviewDataTest = review_dataset[int(len(review_dataset)*0.9):]


# In[25]:


usersPerItem = defaultdict(set) # Maps an item to the users who rated it
itemsPerUser = defaultdict(set) # Maps a user to the items that they rated
itemNames = {}
ratingDict = {} # To retrieve a rating for a specific user/item pair
reviewsPerUser = defaultdict(list)

for d in reviewDataTrain:
    user,item = d['customer_id'], d['product_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    reviewsPerUser[user].append(d)

for d in review_dataset:
    user,item = d['customer_id'], d['product_id']
    ratingDict[(user,item)] = d['star_rating']
    itemNames[item] = d['product_title']


# In[ ]:





# In[30]:


def testQ5():
    a = [x[0] for x in homework2.mostSimilar("B00KCHRKD6", 10, usersPerItem)]
    b = [x[0] for x in reference.mostSimilar("B00KCHRKD6", 10, usersPerItem)]
    return countRight(a,b, 0.01)


# In[31]:


#testQ5()


# In[32]:


ratingMean = homework2.getMeanRating(reviewDataTrain)

userAverages = homework2.getUserAverages(itemsPerUser, ratingDict)

itemAverages = homework2.getItemAverages(usersPerItem, ratingDict)


# In[35]:





# In[42]:


def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    
    simPredictions_ = [reference.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    
    labels = [d['star_rating'] for d in reviewDataTest]
    
    print("Testing (a) is the MSE right; (b) are the predictions right")
    
    a = countRight([homework2.MSE(simPredictions, labels)], [reference.MSE(simPredictions_, labels)], 0.02)
    b = countRight(simPredictions[:50], simPredictions_[:50], 0.02)
    return 0.5 * (a + b)


# In[43]:


#testQ6()


# In[ ]:


def testQ7():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    

    q7Predictions = [homework2.predictRatingQ7(d['customer_id'],
                                               d['product_id'],
                                               ratingMean,
                                               reviewsPerUser,
                                               usersPerItem,
                                               itemsPerUser,
                                               userAverages,
                                               itemAverages) for d in reviewDataTest]
    
    labels = [d['star_rating'] for d in reviewDataTest]
    
    m1 = homework2.MSE(simPredictions, labels)
    m2 = homework2.MSE(q7Predictions, labels)
    m3 = homework2.MSE(alwaysPredictMean, labels)
    
    return 1.0 * ((m2 < m1) and (m2 < m3))


# In[ ]:





--- END of autograder.py ---


In [ ]:
def testQ1():
    mod, validBER, testBER = homework2.Q1(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [ ]:
testQ1()

In [ ]:
def testQ2():
    mod, validBER, testBER = homework2.Q2(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [ ]:
#testQ2()

In [ ]:
def testQ3():
    mod, validBER, testBER = homework2.Q3(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [ ]:
#testQ3()

In [ ]:
def testQ4():
    testBER_noCat, testBER_noReview, testBER_noLength = homework2.Q4(1, catID, dataTrain, dataValid, dataTest)
    return testBER_noCat, testBER_noReview, testBER_noLength

In [ ]:
#testQ4()

In [3]:
import gzip
path = "./amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz"
f = gzip.open(path, 'rt', encoding="utf8")

header = f.readline()
header = header.strip().split('\t')

In [ ]:
header

In [4]:
review_dataset = []

pairsSeen = set()

for line in f:
    fields = line.strip().split('\t')
    d = dict(zip(header, fields))
    ui = (d['customer_id'], d['product_id'])
    if ui in pairsSeen:
        print("Skipping duplicate user/item:", ui)
        continue
    pairsSeen.add(ui)
    d['star_rating'] = int(d['star_rating'])
    d['helpful_votes'] = int(d['helpful_votes'])
    d['total_votes'] = int(d['total_votes'])
    review_dataset.append(d)

Skipping duplicate user/item: ('46953315', 'B00QM3CNN6')
Skipping duplicate user/item: ('31616428', 'B0026RB0G8')
Skipping duplicate user/item: ('47240912', 'B008I653SC')
Skipping duplicate user/item: ('14503091', 'B003FRMRC4')
Skipping duplicate user/item: ('38538360', 'B00HVLUR86')
Skipping duplicate user/item: ('43448024', 'B00HVLUR86')
Skipping duplicate user/item: ('51525270', 'B00HVLUR86')
Skipping duplicate user/item: ('20652160', 'B004OU2IQG')
Skipping duplicate user/item: ('10964440', 'B00HVLUR86')
Skipping duplicate user/item: ('20043677', 'B00HVLUR86')
Skipping duplicate user/item: ('44796499', 'B00HVLUSGM')
Skipping duplicate user/item: ('29066899', 'B0002CZSYO')
Skipping duplicate user/item: ('10385056', 'B004OU2IQG')
Skipping duplicate user/item: ('1658551', 'B00HVLURL8')
Skipping duplicate user/item: ('907433', 'B00N9Q2E5G')
Skipping duplicate user/item: ('39412969', 'B00HVLUR86')
Skipping duplicate user/item: ('4901688', 'B00HVLUR86')
Skipping duplicate user/item: ('234

In [5]:
reviewDataTrain = review_dataset[:int(len(review_dataset)*0.9)]
reviewDataTest = review_dataset[int(len(review_dataset)*0.9):]

In [7]:
from collections import defaultdict
usersPerItem = defaultdict(set) # Maps an item to the users who rated it
itemsPerUser = defaultdict(set) # Maps a user to the items that they rated
itemNames = {}
ratingDict = {} # To retrieve a rating for a specific user/item pair
reviewsPerUser = defaultdict(list)

for d in reviewDataTrain:
    user,item = d['customer_id'], d['product_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    reviewsPerUser[user].append(d)

for d in review_dataset:
    user,item = d['customer_id'], d['product_id']
    ratingDict[(user,item)] = d['star_rating']
    itemNames[item] = d['product_title']

In [33]:
for i in list(usersPerItem.keys())[:5]:
    print(i, usersPerItem[i])

B00HH62VB6 {'34640030', '46877855', '35042585', '1277194', '1420408', '32195459', '4763780', '30694393', '37940864', '6233032', '12255937', '52477340', '25866523', '35102419', '762682', '18426108', '49176182', '33076861', '5852157', '51875431', '11425616', '10525408', '11379232', '7620916', '28194730', '47526811', '46238263', '52410946', '41202145', '49338259', '10299512', '46780231', '23830937', '10917355', '2073579', '24887214', '36979893', '35818167', '2521903', '43091175', '1715524', '32830668', '15362947', '26436380', '36305368', '50414278', '37306273', '49016270', '28891416', '15422113', '43198347', '23689909', '6235503', '8912670', '50230763', '35977706', '52318729', '29131730', '14676111', '17284499', '20618616', '16234687', '17876180', '445857', '11198723', '34657545', '23522394', '42599336', '45137722', '1080628', '45610553', '27894269', '16209753', '12470572', '28850509', '10981888', '28541453', '40614206', '593266', '12863984', '34800941', '6132369'}
B003LRN53I {'23338994',

In [8]:
def Jaccard(s1, s2):
    # Implement |s1 & s2|/ |s1 V s2|
    common = len(s1 & s2)
    both = len(s1 | s2)
    try:
        result = common/both
    except:
        print("sets s1 and s2 empty")
        result = 0
    return result

def mostSimilar(i, N, usersPerItem):
    # initalize list to store similarity results (similarity, itemID)
    similarities = []
    s1 = usersPerItem[i]
    for item in usersPerItem.keys():
        s2 = usersPerItem[item]
        if s1 == s2:
            continue
        jac = Jaccard(s1,s2)
        similarities.append((jac,item))
    # sort similarites
    similarities.sort(reverse=True)
    return(similarities[:N])

mostSimilar("B00KCHRKD6", 10, usersPerItem)




[(0.015228426395939087, 'B00H7NFDKA'),
 (0.014492753623188406, 'B00QKVV3HC'),
 (0.014492753623188406, 'B00GXRMD7W'),
 (0.014084507042253521, 'B00H7ILRRI'),
 (0.014084507042253521, 'B0057RUMPO'),
 (0.014084507042253521, 'B000B6DTYW'),
 (0.013888888888888888, 'B00L2708TI'),
 (0.013513513513513514, 'B009Z1KKWI'),
 (0.013513513513513514, 'B000VYINCW'),
 (0.013333333333333334, 'B003F2BDZQ')]

In [10]:
def getMeanRating(dataTrain):
    mean_rating = np.mean(np.array([d['star_rating'] for d in dataTrain]))
    return(mean_rating)

def getUserAverages(itemsPerUser, ratingDict):
    # Implement (should return a dictionary mapping users to their averages)
    #return userAverages
    userAverages = {}
    for user in itemsPerUser.keys():
        ratings = [ratingDict[(user, item)] for item in itemsPerUser[user]]
        userAverages[user] = np.mean(ratings)
    return(userAverages)

def getItemAverages(usersPerItem, ratingDict):
    # Implement...
    itemAverages = {}
    for item in usersPerItem.keys():
        ratings = [ratingDict[(user, item)] for user in usersPerItem[item]]
        itemAverages[item] = np.mean(ratings)
    return(itemAverages)

def predictRating(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    # Solution for Q6, should return a rating
    def precompute_jaccard_similarity_numpy(usersPerItem):
        items = list(usersPerItem.keys())
        num_items = len(items)
    
        # create the item_id -> index mapping
        item_to_index_map = {item_id: i for i, item_id in enumerate(items)}
    
        # initialize matrix
        similarity_matrix = np.zeros((num_items, num_items), dtype=float)
    
        # fill similarity matrix
        for i in range(num_items):
            for j in range(i, num_items): # Only compute upper triangle
                item1 = items[i]
                item2 = items[j]
            
                s1 = usersPerItem[item1]
                s2 = usersPerItem[item2]
            
                sim = Jaccard(s1, s2)
            
                # store value
                similarity_matrix[i, j] = sim
                similarity_matrix[j, i] = sim
                
        return similarity_matrix, item_to_index_map
    
    similarity_matrix, item_to_index_map = precompute_jaccard_similarity_numpy(usersPerItem)
    ratings = []
    similarities = []
    
    # get R_i_bar
    baseline = itemAverages.get(item,ratingMean)
    
    # get the matrix index for our target item 'i'
    idx_i = item_to_index_map[item]

    # 3. Loop over items j rated by user (j = i2)
    #if user not in reviewsPerUser:
        # User is unknown, return item baseline
        #return baseline
        
    for d in reviewsPerUser[user]:
        i2 = d['product_id'] # This is item 'j'
        
        # Skip if j is the same as our target item i
        if i2 == item: 
            continue
            
        # 4. Get deviation: (R_u,j - R_j_bar)
        try:
            r_j_bar = itemAverages[i2]
        except KeyError:
            continue # This item has no average, skip it
            
        rating_deviation = d['star_rating'] - r_j_bar
        ratings.append(rating_deviation)
        
        # 5. Get similarity: Sim(i, j)
        # This is now a fast NumPy array lookup
        try:
            # Get the matrix index for item 'j'
            idx_j = item_to_index_map[i2]
            sim_ij = similarity_matrix[idx_i, idx_j]
        except KeyError:
            # item 'i2' was not in training, so similarity is 0
            sim_ij = 0
            
        similarities.append(sim_ij)

    # 6. Calculate final prediction
    # Convert lists to NumPy arrays for vectorized operations
    sims_arr = np.array(similarities)
    ratings_arr = np.array(ratings)
    
    # Calculate the sum of similarities (the denominator)
    sims_sum = np.sum(sims_arr)

    # Handle division by zero edge case
    if (sims_sum > 0):
        # Calculate the dot product for the numerator
        # np.dot(ratings_arr, sims_arr) is equivalent to 
        # sum(weightedRatings) from your original code
        weighted_sum = np.dot(ratings_arr, sims_arr)
        
        weighted_deviation = weighted_sum / sims_sum
        return baseline + weighted_deviation
    else:
        # User hasn't rated any similar items
        return baseline
    


In [11]:
ratingMean = getMeanRating(reviewDataTrain)

userAverages = getUserAverages(itemsPerUser, ratingDict)

itemAverages = getItemAverages(usersPerItem, ratingDict)

NameError: name 'np' is not defined

In [50]:
print(ratingMean)
for u_i in list(userAverages.keys())[:5]:
    print(f"{u_i}: {userAverages[u_i]}")

for item_i in list(itemAverages.keys())[:5]:
    print(f"{item_i}: {itemAverages[item_i]}")

for j in list(usersPerItem.keys())[:5]:
    print(f"{j}: {len(list(usersPerItem[j]))}")

4.264863384353867
45610553: 3.0
14640079: 5.0
6111003: 3.0
1546619: 5.0
12222213: 5.0
B00HH62VB6: 4.2926829268292686
B003LRN53I: 4.134529147982063
B0006VMBHI: 3.946969696969697
B002B55TRG: 4.628571428571429
B00N1YPXW2: 4.507462686567164
B00HH62VB6: 82
B003LRN53I: 223
B0006VMBHI: 264
B002B55TRG: 35
B00N1YPXW2: 201


In [66]:
import numpy as np
lim = 0
def predictRating(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    ratings = []
    sims = []
    # loop over reviews by user user_review_i
    for user_review_i in reviewsPerUser[user]:
        item_id = user_review_i["product_id"]
        # don't include comparision of item against itself
        if item_id == item:
            continue
        # calc (R_u,j - R_j_bar)
        rating_diff = user_review_i['star_rating'] - itemAverages[item_id]
        ratings.append(rating_diff)
        # calc Sim(i, j)
        sim = Jaccard(usersPerItem[item],usersPerItem[item_id])
        if lim < 10:
            print(f"")
        sims.append(sim)

    # if no similar items then return item avg
    if len(sims) == 0: 
        final_rating = itemAverages.get(item,ratingMean)
        return(final_rating)
    # calc sums for non empty case
    else:
        ratings_arr = np.array(ratings)
        sims_arr = np.array(sims)
        sims_sum = np.sum(sims_arr)
        # can't divide by zero
        if sims_sum >0:
            final_rating = itemAverages.get(item,ratingMean) + np.dot(ratings_arr, sims_arr)/sims_sum
            return(final_rating)
        else:
            final_rating = itemAverages.get(item,ratingMean)
            return(final_rating)

def predictRatingQ7(customer_id, product_id, ratingMean, reviewsPerUser, 
                    usersPerItem, itemsPerUser, userAverages, itemAverages):
    """
    Predicts a user's rating for an item using an improved item-based 
    collaborative filtering algorithm.

    This implementation improves on the baseline by:
    1.  Using a more robust baseline estimate that includes both user and 
        item biases:
        b_ui = ratingMean + (user_avg - ratingMean) + (item_avg - ratingMean)
    2.  Predicting the *residual* from this baseline, using neighbor items.
    3.  Applying significance weighting (shrinkage) to the Jaccard similarity 
        to reduce noise from low-support (few common raters) similarities.
    4.  Clipping the final prediction to the valid [1.0, 5.0] rating range.

    Formula:
    r_ui = b_ui + [ SUM_j( (r_uj - b_uj) * S'_ij ) / SUM_j( S'_ij ) ]
    
    Where:
    - b_ui: Baseline estimate for user u, item i
    - b_uj: Baseline estimate for user u, item j
    - r_uj: Actual rating for user u, item j
    - S'_ij: Shrunken Jaccard similarity between items i and j

    Args:
        customer_id (str): The user for whom to predict a rating.
        product_id (str): The item for which to predict a rating.
        ratingMean (float): The global average rating across all reviews.
        reviewsPerUser (dict): {customer_id: [list of review dicts]}
        usersPerItem (dict): {product_id: {set of customer_ids}}
        itemsPerUser (dict): {customer_id: {set of product_ids}}
        userAverages (dict): {customer_id: float_avg_rating}
        itemAverages (dict): {product_id: float_avg_rating}

    Returns:
        float: The predicted rating, clipped to the range [1.0, 5.0].
    """
    
    # --- 1. Calculate Biased Baseline Estimate b_ui ---
    
    # Get user bias (b_u)
    # Default to ratingMean if user is new (cold-start)
    user_avg = userAverages.get(customer_id, ratingMean)
    b_u = user_avg - ratingMean
    
    # Get item bias (b_i)
    # Default to ratingMean if item is new (cold-start)
    item_avg = itemAverages.get(product_id, ratingMean)
    b_i = item_avg - ratingMean
    
    # Calculate baseline estimate
    baseline_pred = ratingMean + b_u + b_i
    
    # --- 2. Handle Cold-Start Cases ---
    
    # Get items rated by this user and users who rated this item
    # Default to empty collections for new users/items
    reviews_for_user = reviewsPerUser.get(customer_id, [])
    users_for_item_i = usersPerItem.get(product_id, set())

    # If user or item is new (no reviews or no raters), 
    # we can't use CF. Return the (clipped) baseline.
    if not reviews_for_user or not users_for_item_i:
        return np.clip(baseline_pred, 1.0, 5.0)

    # --- 3. Collaborative Filtering: Weighted Average of Deviations ---
    
    deviations = []
    similarities = []
    
    # Heuristic shrinkage parameter. Higher k = more shrinkage
    shrinkage_k = 1000

    # Loop over all items 'j' reviewed by the user 'u'
    for review in reviews_for_user:
        item_j_id = review["product_id"]
        
        # Don't compare an item to itself
        if item_j_id == product_id:
            continue
            
        # Get users who rated item 'j'
        users_for_item_j = usersPerItem.get(item_j_id, set())
        if not users_for_item_j:
            continue # Skip if neighbor item 'j' has no raters

        # --- 4. Calculate Shrunken Jaccard Similarity S'_ij ---
        
        # Find common raters
        intersection = users_for_item_i.intersection(users_for_item_j)
        intersection_size = len(intersection)
        
        if intersection_size == 0:
            continue # No similarity

        # Jaccard = |A & B| / |A | B|
        union_size = len(users_for_item_i) + len(users_for_item_j) - intersection_size
        jaccard_sim = intersection_size / union_size
        
        # Apply significance weighting (shrinkage)
        significance_weight = intersection_size / (intersection_size + shrinkage_k)
        weighted_sim = jaccard_sim * significance_weight
        
        if weighted_sim <= 0:
            continue

        # --- 5. Calculate Neighbor's Rating Deviation (r_uj - b_uj) ---
        
        # Get baseline estimate for neighbor item 'j' (b_uj)
        item_j_avg = itemAverages.get(item_j_id, ratingMean)
        b_j = item_j_avg - ratingMean
        baseline_j = ratingMean + b_u + b_j # Note: b_u is for the *current user*
        
        # Get user's actual rating for item 'j'
        rating_j = review['star_rating']
        
        # Calculate deviation from baseline
        deviation = rating_j - baseline_j
        
        # Add to lists for final dot product
        deviations.append(deviation)
        similarities.append(weighted_sim)

    # --- 6. Calculate Final Prediction ---
    
    # If no valid, similar neighbors were found, return the baseline
    if not similarities:
        final_rating = baseline_pred
    else:
        sims_arr = np.array(similarities)
        devs_arr = np.array(deviations)
        
        sims_sum = np.sum(sims_arr)
        
        # Avoid division by zero
        if sims_sum > 0:
            weighted_avg_deviation = np.dot(devs_arr, sims_arr) / sims_sum
            final_rating = baseline_pred + weighted_avg_deviation
        else:
            final_rating = baseline_pred

    # --- 7. Clip and Return ---
    # Ensure prediction is within the valid [1.0, 5.0] range
    if final_rating > 4.3:
        final_rating = 5
    return np.clip(final_rating, 1.0, 5.0)
def MSE(y, ypred):
    y_test = np.array(y)
    y_pred = np.array(ypred)
    mse = np.mean((y_test - y_pred)**2)
    return(mse)
def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    q7Predictions = [predictRatingQ7(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]

    for i in reviewDataTest[:5]:
        print(i)
    
    m1 = MSE(simPredictions, labels)
    m2 = MSE(q7Predictions, labels)
    m3 = MSE(alwaysPredictMean, labels)

    print(f"sim pred mse: {m1}")
    print(f"q7 pred mse: {m2}")
    print(f"just mse: {m3}")
    print(q7Predictions[:5])
    print(labels[:5])
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

testQ6()

In [47]:
def predictRating(user,item,ratingMean,reviewsPerUser,usersPerItem,itemsPerUser,userAverages,itemAverages):
    ratings = []
    similarities = []
    for d in reviewsPerUser[user]:
        i2 = d['product_id']
        if i2 == item: continue
        ratings.append(d['star_rating'] - itemAverages[i2])
        similarities.append(Jaccard(usersPerItem[item],usersPerItem[i2]))
    if (sum(similarities) > 0):
        weightedRatings = [(x*y) for x,y in zip(ratings,similarities)]
        return itemAverages[item] + sum(weightedRatings) / sum(similarities)
    else:
        # User hasn't rated any similar items
        if item in itemAverages:
            return itemAverages[item]
        return ratingMean
def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]

    print(alwaysPredictMean[:10])
    print(simPredictions[:10])
    print(labels[:10])
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

testQ6()

[np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867), np.float64(4.264863384353867)]
[np.float64(4.666666666666667), np.float64(4.325581395348837), np.float64(3.8461538461538463), np.float64(4.033333333333333), np.float64(4.264863384353867), np.float64(4.714285714285714), np.float64(4.295336787564767), np.float64(4.407594936708861), np.float64(5.0), np.float64(3.0)]
[5, 5, 2, 1, 5, 3, 5, 5, 4, 1]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# --- Assume 'df' is your DataFrame ---
#reviewDataTrain = review_dataset[:int(len(review_dataset)*0.9)]
#reviewDataTest = review_dataset[int(len(review_dataset)*0.9):]
df = pd.DataFrame(review_dataset)
# df = df.dropna(subset=['review_body', 'review_headline'])

# --- 1. FEATURE ENGINEERING ---
# Combine text
df = pd.DataFrame(reviewDataTest)
df['full_text'] = df['review_headline'].astype(str) + " " + df['review_body'].astype(str)

# *** NEW: Create length features ***
df['text_length'] = df['full_text'].str.len()
df['headline_length'] = df['review_headline'].str.len()

# 2. Define feature groups
target = 'star_rating'
y = df[target]

text_features = 'full_text'
# *** NEW: Add length features to the numeric list ***
numeric_features = ['helpful_votes', 'total_votes', 'text_length', 'headline_length'] 
categorical_features = ['marketplace', 'vine', 'verified_purchase', 'product_category']

# 3. Create the preprocessing ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        # (name, transformer, columns)
        ('text', TfidfVectorizer(stop_words='english', max_features=2000), text_features),
        ('num', StandardScaler(), numeric_features), # StandardScaler will now scale the length features
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

# 4. Create the full ML pipeline
# (Note: I'm removing PCA as discussed. Ridge handles high-D features well)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0)) # Ridge is excellent for this
])

# 5. Split data, train, and evaluate
# X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)
# model_pipeline.fit(X_train, y_train)
# y_pred = model_pipeline.predict(X_test)
# ... etc ...

In [ ]:
ratingMean = homework2.getMeanRating(reviewDataTrain)

userAverages = homework2.getUserAverages(itemsPerUser, ratingDict)

itemAverages = homework2.getItemAverages(usersPerItem, ratingDict)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge # Using Ridge for a stable regression

# --- Assume 'df' is your DataFrame loaded from the dictionaries ---
# df = pd.DataFrame(your_list_of_dicts)
df = pd.DataFrame(review_dataset)
# df = df.dropna(subset=['review_body', 'review_headline']) # Drop rows with no text

# 1. Define feature groups
# We'll combine headline and body for a richer text feature
df['full_text'] = df['review_headline'] + " " + df['review_body']
df['text_length'] = df['full_text'].str.len()
# Define our target (y) and features (X)
target = 'star_rating'
y = df[target]

# Define which columns to use for which transformation
# Note: We drop all ID-like fields
numeric_features = ['helpful_votes', 'total_votes','text_length']
categorical_features = ['marketplace', 'vine', 'verified_purchase', 'product_category']

# 2. Create the preprocessing 'ColumnTransformer'
# This applies different transforms to different columns
preprocessor = ColumnTransformer(
    transformers=[
        # (name, transformer, columns)
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop' # Drop any columns not specified
)

# 3. Create the full ML pipeline
# This chains the preprocessor, PCA, and the final regressor
pcr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=100)), # Start with 100 components
    ('regressor', Ridge(alpha=1.0)) # Use a Ridge regressor for stability
])

# 4. Split data, train, and evaluate
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)

# Fit the entire pipeline on the training data
pcr_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pcr_pipeline.predict(X_test)

# You would then evaluate 'y_pred' against 'y_test' using MSE
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

In [78]:
labels = [d['star_rating'] for d in reviewDataTest]
import matplotlib.pyplot as plt

for i in reviewDataTest[:1]:
    for j in i.keys():
        print(f"{j}: {i[j]}")

marketplace: US
customer_id: 35391331
review_id: R1LJEXO7XBNE6G
product_id: B001WAKLMY
product_parent: 985677168
product_title: Red Witch Analog Famulus Distortion
product_category: Musical Instruments
star_rating: 5
helpful_votes: 0
total_votes: 0
vine: N
verified_purchase: Y
review_headline: Truly unique tone creation
review_body: I got this overdrive since I was looking for something with a wide range of usable sounds but still all analog (I am not a fan of digital effects). The Red Witch Famulus fits the bill perfectly, offering two distinct overdrive circuits in one pedal, and the best part is that you don't have to use one or the other as you can blend them to any ratio you desire. I like to put one of the gains on full and the other pretty low to get a mix of clarity and bite in one tone. If you want to you can crank both for a pretty full hard rock tone or just the first for more biting sound, or whatever you desire really, mixing in the tone (or presence as it may be according

In [ ]:
def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]

    print(alwaysPredictMean[:10])
    print(simPredictions[:10])
    print(labels[:10])

    
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

testQ6()

In [ ]:
#testQ6()

In [ ]:
def testQ7():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    

    q7Predictions = [homework2.predictRatingQ7(d['customer_id'],
                                               d['product_id'],
                                               ratingMean,
                                               reviewsPerUser,
                                               usersPerItem,
                                               itemsPerUser,
                                               userAverages,
                                               itemAverages) for d in reviewDataTest]
    
    labels = [d['star_rating'] for d in reviewDataTest]
    
    m1 = homework2.MSE(simPredictions, labels)
    m2 = homework2.MSE(q7Predictions, labels)
    m3 = homework2.MSE(alwaysPredictMean, labels)
    
    # Autograder checks whether your solution is better than either the Q6 or a naive solution
    return 1.0 * ((m2 < m1) and (m2 < m3))